# Rover on Rough Terrain — Planning Challenge (C++ edition)

A point rover lives on a 2D arena. It must drive from **start** to **goal**.

This is the **C++** port of the rover planning challenge. It runs in Google
Colab using the standard pattern: each source file is written to disk with
`%%writefile`, compiled with `g++`, and the resulting binary is run.

You are given (packed into `world.bin` by the Python setup cell from the same
files the Python version uses, then read directly by the C++):
- `field` — a `(H, W, 3)` terrain map with 3 channels you can see:
  - **ch 0 — rocks**: hard obstacles, binary `0` / `1`; lethal, must avoid
  - **ch 1 — mud**: continuous 0..1
  - **ch 2 — slope**: continuous 0..1
- `demos` — a handful of **expert trajectories** (lists of `(x, y)` points).
  Each demo runs between its *own* start/goal pair (NOT necessarily yours), so
  no single demo is the answer — they're there to reveal the terrain cost.
- `start`, `goal` — the 2D points YOUR path must connect (`x = column`, `y = row`).

**Motion model:** the rover is a point in continuous `(x, y)` space; your path
is any polyline of waypoints (not restricted to a grid). The metric samples
along every segment, so sparse waypoints can't skip over a rock or costly cell.

Your job: a path that reaches the goal and is about as terrain-cheap as the
experts'. A pre-provided baseline cost lets you run a planner immediately, but
its weights are wrong — so plan under it and you'll cost too much. The demos
reveal what the experts actually avoid.

**Two TODOs:** (1) implement the planner, (2) learn the cost from the demos.
Partial progress is fine — talk through your choices as you go.

### Setup &mdash; clone data + shipped C++ sources, pack `world.bin`, import helpers

In [ ]:
# Clone the data repo when running on Colab (no-op if files already exist),
# then PACK everything the C++ side needs into a single flat `world.bin`.
# Letting numpy load the .npy/.npz here (and handle the pickled demos object
# array) means the C++ side has no file-format parsing at all -- it just reads
# this one binary. The layout below is the single shared contract between the
# Python loader and the C++ `load_world()`. Python is used ONLY for this data
# handoff, the rendered-image display, and the matplotlib plotting helper; all
# challenge logic -- metric, planner, learned cost -- lives in C++.
import os, subprocess, struct
import numpy as np
if not os.path.exists('field.npy'):
    subprocess.run(
        ['git', 'clone', '--depth', '1',
         'https://github.com/vvanirudh/planning_challenge.git', '_data'],
        check=True)
    os.chdir('_data')

field = np.load('field.npy').astype(np.float32)          # (H, W, 3)
demos = np.load('demos.npy', allow_pickle=True)          # object array of (T,2)
meta  = np.load('meta.npz')
H, W = field.shape[:2]

# ---- world.bin layout (all little-endian) --------------------------------
#   int32 H, int32 W
#   H*W*3 float32   field, row-major [r, c, ch]
#   H*W   float64   true_cost (rock cells carry the sentinel)
#   double start_x, start_y, goal_x, goal_y
#   double expert_cost, tol, rock_sentinel
#   int32 n_demos; per demo: int32 T, then T*2 float64 (x, y interleaved)
with open('world.bin', 'wb') as f:
    f.write(struct.pack('<ii', H, W))
    f.write(np.ascontiguousarray(field, np.float32).tobytes())
    f.write(np.ascontiguousarray(meta['true_cost'], np.float64).tobytes())
    f.write(np.asarray(meta['start'], np.float64).tobytes())
    f.write(np.asarray(meta['goal'],  np.float64).tobytes())
    f.write(struct.pack('<ddd', float(meta['expert_cost']), float(meta['tol']),
                        float(meta['rock_sentinel'])))
    f.write(struct.pack('<i', len(demos)))
    for d in demos:
        d = np.asarray(d, np.float64)
        f.write(struct.pack('<i', d.shape[0]))
        f.write(np.ascontiguousarray(d, np.float64).tobytes())

# The prebuilt C++ plumbing (terrain.hpp / metric.hpp / baseline.hpp, the
# main_*.cpp drivers) and the matplotlib helper viz.py ship in this repo and are
# now on disk in the working dir. Import the two Python helpers from viz.py.
import sys
sys.path.insert(0, os.getcwd())
from viz import show_cpp, build_and_run

print('g++ version:',
      subprocess.run(['g++', '--version'], capture_output=True, text=True).stdout.splitlines()[0])
print(f'packed world.bin: field {field.shape} | demos {len(demos)} '
      f'| start {meta["start"]} goal {meta["goal"]}')
# Compile-check the shipped driver that needs no candidate code, so a bad clone
# or toolchain surfaces here rather than inside a TODO.
_chk = subprocess.run(['g++', '-O2', '-std=c++17', 'main_show.cpp', '-o', '_chk'],
                      capture_output=True, text=True)
print('toolchain + shipped sources:', _chk.stderr.strip() or 'OK')

### The API you have (do not edit — it ships in the cloned repo)

All the plumbing — `terrain.hpp` (data + loader + `show`), `metric.hpp`
(`path_cost`/`evaluate`), `baseline.hpp`, the `main_*.cpp` drivers, and the
Python `viz.py` (`show_cpp`, `build_and_run`) — is on disk after the setup cell.
**You edit only the two `%%writefile` cells below: `planner.hpp` and
`learn.hpp`.** Re-running a TODO cell rewrites that header; the next
`build_and_run` recompiles and picks up your edit.

**`World` (passed to everything as `w`)**
- `w.H`, `w.W` — grid size (ints).
- `w.start`, `w.goal` — `Vec2{double x, y}`, with `x = column`, `y = row`.
- `w.demos` — `std::vector<Path>`, where `Path = std::vector<Vec2>`. Expert
  polylines, each with its *own* endpoints (not yours).
- `w.fld(r, c, ch) -> float` — terrain at row `r`, col `c`, channel
  `ROCK`/`MUD`/`SLOPE` (= 0/1/2). `ROCK` is binary & lethal; `MUD`/`SLOPE` ∈ 0..1.
- (`w.true_cost` is the hidden scoring grid the metric reads — it's in `World`
  so `evaluate` can use it, but it's off-limits to you: using it is the thing
  you're meant to recover from the demos, and it won't generalize past this map.)

**Cost maps** are `std::vector<double>` of length `H*W`, row-major:
`cost_map[r*W + c]`.

**You implement (in the TODO cells):**
- `Path plan(const World& w, const std::vector<double>& cost_map, int K=256, int T=60, int iters=30, unsigned seed=0)` — a start→goal polyline.
- `std::vector<double> cost_map_learned(const World& w)` — a row-major `H*W` cost built from `w.demos`.

**Prebuilt (call these):**
- `cost_map_baseline(const World& w) -> std::vector<double>` — wrong-but-runnable cost (`1 + 5*slope`, rocks lethal).
- `path_cost(const World& w, const Path& p, const std::vector<double>& cost_map) -> double` — integrate *your* cost along a path; lower is better. Score rollouts with this.
- `evaluate(const World& w, const Path& p, const std::vector<double>& cost_map) -> bool` — official PASS/FAIL scorer; prints `start_ok/goal_ok/model_feasible/rock_hit/out_of_bounds/cost/bar`. PASS needs endpoints near start/goal, feasible under `cost_map`, in bounds, no rock under the hidden true cost, and terrain cost ≤ the expert bar. The bar has comfortable tolerance — an expert-like path that avoids the costly terrain clears it, so treat this PASS/FAIL line as your feedback signal and iterate against it rather than chasing an exact cost match.

The true terrain cost is hidden behind the metric — recovering it from the demos
is the point. The full source is in the cloned repo if you want to read it.

### Initial visualization &mdash; the true cost map + expert demos (gray)

In [ ]:
build_and_run('main_show.cpp', [('initial.bin', 'true cost map + expert demos (gray)')])

## TODO 1 &mdash; implement the planner (`plan`); details in the cell

In [ ]:
%%writefile planner.hpp
// ---- TODO 1: a sampling-based rollout planner (CPU first) ----------------
// Plan a cheap start->goal path under a given (H,W) cost_map. Suggested scheme
// (you may do otherwise): hold a current T-waypoint trajectory; each iteration
// sample K noisy rollouts around it, score each with path_cost(w, p, cost_map),
// and step toward the cheap ones (softmax / take-best).
#pragma once
#include "metric.hpp"
#include "terrain.hpp"
#include <vector>

inline Path plan(const World& w, const std::vector<double>& cost_map,
                 int K = 256, int T = 60, int iters = 30, unsigned seed = 0) {
    // TODO: replace this straight line (which ignores cost_map) with the
    // sampling rollout loop.
    Path p(T);
    for (int i = 0; i < T; ++i) {
        double a = double(i) / (T - 1);
        p[i] = { w.start.x + (w.goal.x - w.start.x) * a,
                 w.start.y + (w.goal.y - w.start.y) * a };
    }
    return p;
}

### Run the planner under the baseline cost &mdash; expect `FAIL` (wrong weights)

In [ ]:
build_and_run('main_run1.cpp', [('run1.bin', 'baseline cost: planned path')])

## TODO 2 &mdash; learn the cost from demos (`cost_map_learned`); details in the cell

In [ ]:
%%writefile learn.hpp
// ---- TODO 2: learn the cost from the expert demos ------------------------
// The baseline charges for the wrong terrain. Use the demos to build a cost map
// that explains the experts' detours, then re-plan. Pick an approach and
// justify it: visitation (cells experts use are cheap), feature regression /
// IRL (fit weights over [mud, slope]), nearest-demo / warping, ...
//
// What "good" means here: your cost only has to RANK terrain like the experts
// do -- cheap where they drive, costly where they detour -- so the planner
// routes the same way. You do NOT need to recover the true cost numerically;
// getting the ordering right (and keeping rocks impassable) is enough. The pass
// bar has comfortable tolerance, so an expert-like path that avoids the costly
// terrain clears it -- don't over-fit chasing an exact cost match.
#pragma once
#include "baseline.hpp"
#include "terrain.hpp"

inline std::vector<double> cost_map_learned(const World& w) {
    // TODO: use w.demos. (Falls back to the wrong baseline for now.)
    return cost_map_baseline(w);
}

### Run the learned solution &mdash; goal: `PASS`

In [ ]:
build_and_run('main_run2.cpp', [('run2_cost.bin', 'learned: planned path over learned cost map'), ('run2_terrain.bin', 'learned: planned path over terrain')])

### Bonus / discussion — make it fast

Your planner rolls out `K` trajectories every iteration. On a real robot we
replan in a closed loop at, say, 50 Hz with `K` in the thousands.

- **Vectorize / parallelize** the rollout: score all `K` candidates without a
  serial loop — e.g. OpenMP over the K rollouts, SIMD on the per-step cost
  lookup, or move the cost gather + sum to CUDA on the GPU.
- Be ready to discuss: memory layout of the `K × T × 2` buffer, how `K` trades
  off against horizon `T`, host↔device transfer / sync cost, and what your
  per-replan time budget buys you at 50 Hz.
- In C++ specifically: where do `float` vs `double`, cache locality of the cost
  grid, and avoiding per-rollout allocations buy you headroom?